# Landmark Maps per Grid Cell

**Goal:** For each 5km control cell, generate maps showing recognizable landmarks —
villages, hamlets, clinics, schools, churches, roads, rivers, etc.

**Approach:** We use pre-rendered **tile basemaps** from different providers to
compare label coverage. Each cell gets three views:
1. **Google Hybrid** — satellite + Google's place labels (best coverage in rural Africa)
2. **Esri Satellite + labels overlay** — Esri WorldImagery + CartoDB labels on top
3. **CartoDB Voyager** — OSM-based street map with all labels (for reference)

No API queries needed — just tile downloads.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx

from src.utils.config_loader import load_config, get_data_dir
from src.data_processing.load_boundaries import load_control_grid

config = load_config()
data_dir = get_data_dir(config)
print(f"Data directory: {data_dir}")

In [ ]:
# ===== TEST MODE: set to True to only process first N cells =====
TEST_MODE = True
N_TEST_CELLS = 5
# =================================================================

# Load 5km control grid
grid_5km = load_control_grid(data_dir)
print(f"Control cells: {len(grid_5km)}")

# Subset for test mode
if TEST_MODE:
    grid_5km = grid_5km.head(N_TEST_CELLS).copy()
    print(f"\n** TEST MODE: showing first {N_TEST_CELLS} cells only **")

# Project to web mercator for plotting with basemaps
grid_wm = grid_5km.to_crs(epsg=3857)
print(f"Plotting {len(grid_5km)} cells")
print(f"Cell IDs: {grid_5km['id'].tolist()}")

## Maps: comparing label providers per grid cell

Three columns per cell to compare coverage:
1. **Google Hybrid** — satellite imagery with Google's labels overlaid (typically
   best coverage in rural Africa, similar to Bing Maps)
2. **Esri + labels** — Esri WorldImagery with a transparent CartoDB labels layer on top
3. **CartoDB Voyager** — pure OSM-based map with all labels (baseline/reference)

In [ ]:
# Basemap tile providers
import xyzservices

GOOGLE_HYBRID = xyzservices.TileProvider(
    url="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
    name="Google Hybrid",
    attribution="(C) Google",
)
ESRI_SATELLITE = cx.providers.Esri.WorldImagery
CARTO_LABELS = cx.providers.CartoDB.PositronOnlyLabels  # transparent labels-only layer
CARTO_VOYAGER = cx.providers.CartoDB.Voyager

ZOOM = 14  # good detail level for 5km cells

# Plot each cell: 3 columns
cell_ids = grid_5km["id"].values
n_cells = len(cell_ids)

fig, axes = plt.subplots(n_cells, 3, figsize=(20, 6 * n_cells))
if n_cells == 1:
    axes = axes.reshape(1, 3)

for row_idx, cell_id in enumerate(cell_ids):
    cell_wm = grid_wm[grid_wm["id"] == cell_id]
    minx, miny, maxx, maxy = cell_wm.total_bounds
    buf = (maxx - minx) * 0.05

    # --- Col 1: Google Hybrid (satellite + labels) ---
    ax = axes[row_idx, 0]
    cell_wm.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=3)
    ax.set_xlim(minx - buf, maxx + buf)
    ax.set_ylim(miny - buf, maxy + buf)
    cx.add_basemap(ax, source=GOOGLE_HYBRID, zoom=ZOOM)
    ax.set_title(f"Cell {cell_id} — Google Hybrid", fontsize=11)
    ax.set_axis_off()

    # --- Col 2: Esri satellite + CartoDB labels overlay ---
    ax = axes[row_idx, 1]
    cell_wm.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=3)
    ax.set_xlim(minx - buf, maxx + buf)
    ax.set_ylim(miny - buf, maxy + buf)
    cx.add_basemap(ax, source=ESRI_SATELLITE, zoom=ZOOM)
    cx.add_basemap(ax, source=CARTO_LABELS, zoom=ZOOM)
    ax.set_title(f"Cell {cell_id} — Esri + labels", fontsize=11)
    ax.set_axis_off()

    # --- Col 3: CartoDB Voyager (OSM reference) ---
    ax = axes[row_idx, 2]
    cell_wm.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=3)
    ax.set_xlim(minx - buf, maxx + buf)
    ax.set_ylim(miny - buf, maxy + buf)
    cx.add_basemap(ax, source=CARTO_VOYAGER, zoom=ZOOM)
    ax.set_title(f"Cell {cell_id} — CartoDB Voyager (OSM)", fontsize=11)
    ax.set_axis_off()

plt.tight_layout()
plt.show()

## Next steps

- Review the OSM labeled maps to identify which villages/landmarks are inside each cell
- Cells with visible village names + amenities on the Voyager map → enumerators go to village office
- Cells with sparse OSM labels → may need random walk, or cross-check with Google Maps
- Consider saving these maps as PDFs/images for field team reference
- Consider overlaying landmarks on the sub-cell sampling maps from notebook 06